# A1.8 · Malicious code execution

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.7 · Identity spoofing and impersonation](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**.

| | |
|---|---|
| Tools used | Falco, gVisor, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Execute model-authored code and enumerate what the process could touch.

**Why a security engineer needs it.** Model-authored code runs with the runtime's privileges — reaching the filesystem, the network and any credential in the environment. The control it builds is: sandboxed execution (A3.2) and egress control (A3.3).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Asking a model to write code is safe. Running the code it wrote is the part that is not, and most agent frameworks ship the second one enabled with the same process privileges as the framework itself.

> **At CyberTravels.** The Coding Agent writes a patch and the runtime executes it. On Alex's laptop that process can read `~/.aws`, the HR folder and the roadmap directory, because nothing said otherwise. R6.

## 2 · The framework

```
   model ---> "here is a script that does it" ---> agent runtime
                                                        |
                                            exec() on the host
                                                        v
                                        whatever the PROCESS can reach:
                                        files . network . credentials . socket

   writing the code is safe. running it is the part that is not.
```

**OWASP T11 — Unexpected RCE and Code Attacks. LLM05 — Improper Output Handling.**

Many useful agents write code and run it — that is what makes a data-analysis
agent or a coding agent worth having. The **agent_runtime** component executes
text the **model** produced, on a host, in a process.

The risk is not exotic. Model-authored code is just code, and it runs with
whatever the process has: the filesystem it can see, the network it can reach,
and every credential in its environment. There is no privilege boundary between
"the code the agent wrote to reformat a CSV" and "the code that reads
`~/.aws/credentials`", because both are strings passed to the same interpreter.

Two paths lead here, and only one involves an attacker:

**Steered.** An injection from A1.3 tells the agent to write particular code.
The runtime executes it because executing code is its job.

**Unsteered.** Nobody attacked anything. The agent wrote something plausible and
wrong — a cleanup routine with a path variable that resolves higher than
intended — and the blast radius was decided by the environment, not by intent.

That second path is worth sitting with. Most teams model this as an attack. In
practice the first incident is usually an ordinary bug with production
credentials in scope, which is why the control in A3.2 is about what the process
can *reach*, not about what the model can be persuaded to *write*.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

What the executing process can reach, enumerated rather than assumed. Nothing below actually touches your machine — the environment is a fixture, so the lesson runs anywhere.

## Your turn

For one agent that executes code, list what is in its process environment right now. The credentials in that list are the blast radius of the next ordinary bug, not of the next attack.

---

**Next → [A1.9 · Injection through content the agent was asked to read](https://spbreed.github.io/cyber-commons/lessons/A1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*